Нотбук выполняет первый этап пайплайна:
1. Подключение к Telegram API (через библиотеку Telethon).
2. Сбор сообщений с заданных каналов.
3. Первичную фильтрацию пустых сообщений.
4. Применение функции глубокой лингвистической очистки текста.
5. Расчет уникальных SHA256-хэшей очищенных текстов для устранения дубликатов.
6. Экспорт собранных данных в файл `data_raw.csv`.

In [ ]:
import os
import re
import hashlib
import pandas as pd
import warnings
import nltk
from nltk.corpus import stopwords
from typing import List, Dict, Any

from dotenv import load_dotenv
from telethon import TelegramClient
from python_socks import ProxyType

warnings.filterwarnings("ignore")

# Настройки отображения Pandas
pd.set_option("display.max_columns", 10)
pd.set_option("display.max_colwidth", 150)

# Директория для сохранения результатов
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
RAW_DATA_PATH = os.path.join(DATA_DIR, "data_raw.csv")

In [ ]:
# Безопасная фоновая загрузка стоп-слов NLTK
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords", quiet=True)

RU_STOPWORDS = set(stopwords.words("russian"))
EN_STOPWORDS = set(stopwords.words("english"))
ALL_STOPWORDS = RU_STOPWORDS.union(EN_STOPWORDS)

def clean_raw_text(text: str) -> str:
    """
    Выполняет глубокую очистку текста для технического поля 'text':
    - Удаляет веб-ссылки, Telegram-ссылки, телефоны и юзернеймы.
    - Переводит все знаки конца предложения (!, ?) в точки.
    - Удаляет всю пунктуацию, кроме букв, цифр и точек.
    - Фильтрует русские и английские стоп-слова.
    - Сохраняет одиночные точки (.) для предотвращения склеивания слов на границах предложений.
    """
    text_str = str(text).lower()

    # 1. Удаление веб-ссылок, ссылок на Telegram-каналы/сообщения и телефонных номеров
    text_str = re.sub(r"https?://\S+|www\.\S+", " ", text_str)
    text_str = re.sub(r"\bt\.me/\S+|@\S+", " ", text_str)
    text_str = re.sub(r"(\+?\d[\d\s\-\(\)]{7,}\d)", " ", text_str)

    # 2. Превращение знаков конца предложения в единую точку для сохранения границ
    text_str = re.sub(r"[!?\n\r\t]+", ". ", text_str)

    # 3. Удаление всех спецсимволов и пунктуации, кроме кириллицы, латиницы, цифр и точек
    text_str = re.sub(r"[^a-zа-яё0-9.\s]", " ", text_str)

    # 4. Пословная фильтрация стоп-слов с сохранением семантических точек
    raw_tokens = text_str.split()
    cleaned_tokens: List[str] = []

    for token in raw_tokens:
        # Обрабатываем токен, если он является самостоятельной точкой
        if token == ".":
            if not cleaned_tokens or cleaned_tokens[-1] != ".":
                cleaned_tokens.append(".")
            continue

        # Обрабатываем слова, которые заканчиваются на точку (конец предложения)
        if token.endswith("."):
            clean_word = token[:-1].strip()
            if clean_word and clean_word not in ALL_STOPWORDS and not clean_word.isdigit():
                cleaned_tokens.append(clean_word)
            if not cleaned_tokens or cleaned_tokens[-1] != ".":
                cleaned_tokens.append(".")
            continue

        # Обрабатываем стандартные слова
        if token not in ALL_STOPWORDS and not token.isdigit():
            cleaned_tokens.append(token)

    # Склеиваем слова обратно в текст
    result_text = " ".join(cleaned_tokens)
    
    # Лингвистическое форматирование: убираем пробел перед точкой ("слово ." -> "слово.")
    result_text = re.sub(r"\s+\.", ".", result_text)
    
    # Схлопываем множественные точки в одну ("слово..." -> "слово.")
    result_text = re.sub(r"\.+", ".", result_text)
    
    # Удаляем лишние пробелы
    result_text = re.sub(r"\s+", " ", result_text).strip()

    return result_text


def calculate_sha256(text: str) -> str:
    """
    Вычисляет хэш-сумму SHA256 для строки.
    """
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

In [ ]:
# Определение списка каналов для парсинга
CHANNELS = {
    "digital_jobster": "https://t.me/digital_jobster",
    "vdhl_good": "https://t.me/vdhl_good",
    "designhunters": "https://t.me/designhunters",
}

# Максимальное количество сообщений с одного канала
LIMIT = 100000

# Настройки прокси
USE_PROXY = True
PROXY_HOST = "127.0.0.1"
PROXY_PORT = 12334

In [ ]:
async def parse_channel(client: TelegramClient, channel_name: str, channel_url: str, limit: int = LIMIT) -> List[Dict[str, Any]]:
    """
    Парсит один Telegram-канал и возвращает сообщения, содержащие текст.
    """
    print(f"Парсинг канала: {channel_name} ({channel_url})")
    channel_posts = []

    try:
        entity = await client.get_entity(channel_url)

        async for message in client.iter_messages(entity, limit=limit):
            # Пропускаем сообщения без текста
            if not message.text:
                continue

            # Сохраняем согласно алгоритму: id, channel, date, original_text
            channel_posts.append({
                "id": message.id,
                "channel": channel_name,
                "date": message.date.isoformat() if message.date else None,
                "original_text": message.text
            })

        print(f"  -> Успешно получено постов с текстом: {len(channel_posts)}")

    except Exception as error:
        print(f"  -> Ошибка при получении данных из {channel_name}: {error}")

    return channel_posts

In [ ]:
# Загрузка переменных окружения
load_dotenv()

api_id = os.getenv("API_ID")
api_hash = os.getenv("API_HASH")
phone_number = os.getenv("PHONE_NUMBER")
password = os.getenv("PASSWORD")

if not api_id or not api_hash or not phone_number:
    raise ValueError("Пожалуйста, убедитесь, что API_ID, API_HASH и PHONE_NUMBER заданы в файле .env.")

# Настройка прокси
proxy = None
if USE_PROXY:
    proxy = {
        "proxy_type": ProxyType.SOCKS5,
        "addr": PROXY_HOST,
        "port": PROXY_PORT,
        "rdns": True,
    }

# Инициализация клиента Telethon
client = TelegramClient(
    "posts_collector_session",
    int(api_id),
    api_hash,
    device_model="PC",
    system_version="Windows 10",
    app_version="4.15.2",
    lang_code="ru",
    system_lang_code="ru-RU",
    connection_retries=5,
    retry_delay=3,
    timeout=30,
    proxy=proxy,
)

# Запуск парсинга в текущем event loop'е Jupyter
await client.start(phone=phone_number, password=password)
print("Telegram-клиент успешно авторизован.")

# Создание пустого списка posts
posts = []

# Сбор постов со всех каналов
for name, url in CHANNELS.items():
    channel_data = await parse_channel(client, name, url, limit=LIMIT)
    posts.extend(channel_data)

await client.disconnect()
print(f"\nСбор завершен. Общее количество собранных сырых сообщений с текстом: {len(posts)}")

In [ ]:
# Создание DataFrame из всех собранных сообщений
df = pd.DataFrame(posts)

print("Начальный размер датасета:", df.shape)

# Удалить строки с пустым original_text
df = df.dropna(subset=["original_text"]).copy()
df = df[df["original_text"].astype(str).str.strip() != ""].copy()

# Удалить лишние пробелы (по краям текста)
df["original_text"] = df["original_text"].astype(str).str.strip()

# Создать очищенное поле: text = clean_raw_text(original_text)
print("Применение функции глубокой очистки текстов clean_raw_text...")
df["text"] = df["original_text"].apply(clean_raw_text)

# Удалить строки с пустым text
df = df.dropna(subset=["text"]).copy()
df = df[df["text"].astype(str).str.strip() != ""].copy()

# Создать SHA256-хэш: text_hash = sha256(text)
print("Вычисление хэшей")
df["text_hash"] = df["text"].apply(calculate_sha256)

# Удалить дубликаты по text_hash
before_dedup = len(df)
df = df.drop_duplicates(subset=["text_hash"]).copy()
after_dedup = len(df)

print(f"Удалено дубликатов: {before_dedup - after_dedup}")
print("Финальный размер датасета:", df.shape)

# Сохранить: id, channel, date, original_text, text, text_hash в data_raw.csv
export_columns = ["id", "channel", "date", "original_text", "text", "text_hash"]
df_export = df[export_columns].copy()

df_export.to_csv(RAW_DATA_PATH, index=False, encoding="utf-8-sig")
print(f"Данные успешно сохранены в: {RAW_DATA_PATH}")

# Просмотр первых нескольких строк для проверки
df_export.head(5)